# Lab 6.3 &mdash; The Model Interface: Prompts & .invoke()

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Build a PromptTemplate with an {input} slot and format it
- Call the model with .invoke() and read the reply from .content
- See that the SAME interface works for any model (model-agnostic)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; LangChain's core building blocks](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-03"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Every LangChain chat model shares **one interface**: `model.invoke(prompt)` returns a message whose
text is in **`.content`**. That is why swapping **`ChatOllama`** for **`ChatGroq`** is a one-line
change. A **`PromptTemplate`** fills slots like `{input}`. Here a deterministic `FakeChatModel`
stands in for a real model so the lab runs offline.

In [ ]:
class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

demo_model = FakeChatModel(["Hi! I am a stand-in model."])
demo_prompt = PromptTemplate.from_template("Say hi to {who}.")
print(demo_prompt.format(who="Ada"))
print("reply:", demo_model.invoke("anything").content)

## Your Turn
Build a prompt with an `{input}` slot, then `ask` the model and return the reply text.

In [ ]:
def build_prompt(question):
    template = PromptTemplate.from_template(___)   # TODO: a template with an {input} slot
    return ___   # TODO: format the template with input=question

def ask(model, question):
    prompt = build_prompt(question)
    message = ___   # TODO: call the model on the prompt
    return ___      # TODO: the reply text (a message has .content)

try:
    print(build_prompt('what is an agent?'))
    print('---')
    print("reply:", ask(FakeChatModel(["An agent uses tools in a loop."]), "what is an agent?"))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the prompt fills in the question", lambda: "capital of france" in build_prompt("capital of france"))
expect_true("the prompt has a Question: slot", lambda: "Question:" in build_prompt("x"))
expect_true("ask reads the reply from .content", lambda: ask(FakeChatModel(["hi"]), "q") == "hi")
expect_true("the model returns a message with .content", lambda: hasattr(FakeChatModel(["x"]).invoke("p"), "content"))
expect_true("same interface, swappable model", lambda: ask(FakeChatModel(["A"]), "q") == "A" and ask(FakeChatModel(["B"]), "q") == "B")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
One interface -- .invoke(prompt).content -- over every provider. That uniformity is what makes the Ollama<->Groq swap a single line.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>